# Neyshekar v6 — corpus statistics

Statistics and figures for the data descriptor, computed from the Hugging Face release
[`shekar-ai/neyshekar-v6-persian-asr-fa`](https://huggingface.co/datasets/shekar-ai/neyshekar-v6-persian-asr-fa).
Annotation uses [`shekar`](https://doi.org/10.21105/joss.09128); figures go to `figures/` at 300 dpi.

In [ ]:
# Resolve shared code when launched from code/ or the repository root.
import sys
from pathlib import Path

_code_candidates = [Path.cwd(), Path.cwd() / "code"]
CODE = next(
    (
        p.resolve()
        for p in _code_candidates
        if (p / "neyshekar_experiments" / "protocol.py").is_file()
    ),
    None,
)
if CODE is None:
    raise RuntimeError("Launch this notebook from the repository root or its code/ directory.")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from neyshekar_experiments.protocol import ROOT
# End notebook bootstrap

import json
import re
import time
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

HF_REPO = "shekar-ai/neyshekar-v6-persian-asr-fa"
HF_REVISION = "7613a5adebabb5f8f1255a47ef42ca7d7b44046c"


DATA = ROOT / "data"
FIGS = ROOT / "figures"
PAPER_FIGS = ROOT / "acl"
DATA.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)
PAPER_FIGS.mkdir(exist_ok=True)

META_CACHE = DATA / "neyshekar_v6_meta.parquet"
ANNOT_CACHE = ROOT / "results/annotations/neyshekar.parquet"
ANNOT_CACHE.parent.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")

# Categorical hues in fixed order, validated for colour-vision deficiency across
# all pairs rather than eyeballed (worst CVD dE 9.2, normal-vision dE 24.0).
# seaborn's "deep" default fails that check: its orange and green sit dE 4.5
# apart under protanopia, indistinguishable to a red-green colourblind reader.
# Magnitude charts use a single hue, since the axis labels already carry identity.
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE = "#ffffff"
sns.set_palette(CATEGORICAL)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300  # manuscript figures ship at 300 dpi
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.titleweight"] = "semibold"

SPLIT_ORDER = ["train", "validation", "test"]
REGISTER_ORDER = ["formal", "informal"]
REGISTER_COLORS = dict(zip(REGISTER_ORDER, CATEGORICAL[:2]))


def save(fig, name, manuscript_name=None):
    "Write a standalone figure and, when requested, its manuscript copy."
    path = FIGS / f"{name}.png"
    fig.savefig(path)
    print(f"saved {path.relative_to(ROOT)} at {plt.rcParams['savefig.dpi']:.0f} dpi")
    if manuscript_name:
        manuscript_path = PAPER_FIGS / f"{manuscript_name}.png"
        fig.savefig(manuscript_path)
        print(f"saved {manuscript_path.relative_to(ROOT)}")


print(f"figures -> {FIGS.relative_to(ROOT)}")

## 1. Load the corpus metadata

Parquet is columnar, so projecting `id`/`text`/`duration` skips the ~11 GB of audio. Split comes
from the shard filename. Cached to `data/`.

In [ ]:
def load_metadata(
    repo: str = HF_REPO, revision: str = HF_REVISION, cache: Path = META_CACHE
) -> pd.DataFrame:
    "Read id/text/duration/split for every clip, skipping the audio column entirely."
    if cache.exists():
        print(f"loading cached metadata from {cache.relative_to(ROOT)}")
        return pd.read_parquet(cache)

    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    shards = sorted(fs.glob(f"datasets/{repo}@{revision}/data/*.parquet"))
    print(f"reading {len(shards)} shards from {repo} (metadata columns only)")

    frames = []
    for i, shard in enumerate(shards, 1):
        split = re.search(r"/(train|validation|test)-\d+", shard).group(1)
        with fs.open(shard, "rb") as fh:
            frame = pq.read_table(fh, columns=["id", "text", "duration"]).to_pandas()
        frame["split"] = split
        frames.append(frame)
        print(f"  [{i}/{len(shards)}] {Path(shard).name} -> {len(frame):,} rows")

    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(cache, index=False)
    return df


df = load_metadata()
required = {"id", "text", "duration", "split"}
if set(df.columns) != required:
    raise ValueError(f"metadata columns {set(df.columns)!r} do not match {required!r}")
if df[list(required)].isna().any().any():
    raise ValueError("metadata contains missing required values")
if df["id"].duplicated().any() or set(df["id"]) != set(range(len(df))):
    raise ValueError("clip identifiers must be unique and contiguous from zero")
if (df["duration"] <= 0).any():
    raise ValueError("clip durations must be positive")
if not set(df["split"]).issubset(SPLIT_ORDER):
    raise ValueError(f"unexpected split labels: {set(df['split']) - set(SPLIT_ORDER)}")
df["split"] = pd.Categorical(df["split"], categories=SPLIT_ORDER, ordered=True)
df["n_chars"] = df["text"].str.len()

print(f"\n{len(df):,} clips  |  {df.duration.sum() / 3600:.2f} hours")
df.head()

## 2. Corpus overview

In [ ]:
def overview(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "clips": len(frame),
            "hours": frame.duration.sum() / 3600,
            "mean duration (s)": frame.duration.mean(),
            "median duration (s)": frame.duration.median(),
        }
    )


by_split = df.groupby("split", observed=True).apply(overview, include_groups=False)
by_split["share of clips (%)"] = 100 * by_split["clips"] / len(df)

print("Corpus totals")
print(f"  clips              {len(df):,}")
print(f"  total duration     {df.duration.sum() / 3600:.2f} h")
print(f"  mean duration      {df.duration.mean():.2f} s")
print(f"  median duration    {df.duration.median():.2f} s")
print(f"  duration range     {df.duration.min():.2f} - {df.duration.max():.2f} s")
print(
    f"  distinct prompts   {df.text.nunique():,} ({100 * df.text.nunique() / len(df):.1f}% of clips)"
)
print()
by_split.round(2)

## 3. Clip duration

Utterance length is the first design axis, so it is characterised beyond the mean.

In [ ]:
pctl = df.duration.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).rename("seconds")
print("Duration percentiles (s)")
print(pctl.round(2).to_string())
print(
    f"\nstd {df.duration.std():.2f} s  |  IQR {df.duration.quantile(0.75) - df.duration.quantile(0.25):.2f} s"
)
print(f"clips over 10 s: {(df.duration > 10).sum():,} ({100 * (df.duration > 10).mean():.1f}%)")
print(f"clips under 4 s: {(df.duration < 4).sum():,} ({100 * (df.duration < 4).mean():.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.histplot(
    df,
    x="duration",
    bins=np.arange(0, df.duration.max() + 1, 0.5),
    kde=True,
    ax=ax,
    edgecolor="none",
)
ax.axvline(
    df.duration.mean(), color="crimson", ls="--", lw=1.4, label=f"mean {df.duration.mean():.2f} s"
)
ax.axvline(
    df.duration.median(),
    color="darkorange",
    ls=":",
    lw=1.6,
    label=f"median {df.duration.median():.2f} s",
)
ax.legend(frameon=False)
ax.set(xlabel="duration (s)", ylabel="clips")
fig.tight_layout()
save(fig, "02a_duration_distribution", "duration_distribution")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.ecdfplot(df, x="duration", hue="split", ax=ax)
ax.set(xlabel="duration (s)", ylabel="cumulative proportion")
fig.tight_layout()
save(fig, "02b_duration_ecdf_split")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.boxplot(df, x="split", y="duration", hue="split", legend=False, showfliers=False, ax=ax)
ax.set(xlabel="", ylabel="duration (s)")
fig.tight_layout()
save(fig, "02c_duration_boxplot_split", "duration_by_split")
plt.show()
plt.close(fig)

## 4. Register — formal vs informal

`InformalLanguageClassifier` returns `(label, flag)`; flag `1` marks colloquial *goftari*.

In [ ]:
from shekar import InformalLanguageClassifier

informal_clf = InformalLanguageClassifier()
df["register"] = np.where(df["text"].map(lambda t: informal_clf(t)[1] == 1), "informal", "formal")
df["register"] = pd.Categorical(df["register"], categories=REGISTER_ORDER, ordered=True)

reg = df.groupby("register", observed=True).agg(
    clips=("id", "size"),
    hours=("duration", lambda s: s.sum() / 3600),
    mean_duration=("duration", "mean"),
)
reg["share (%)"] = 100 * reg["clips"] / len(df)
print(reg.round(2).to_string())
print()
print("Register balance within each split (% of clips)")
print((pd.crosstab(df["split"], df["register"], normalize="index") * 100).round(2).to_string())

## 5. Tokens and lexicon

`WordTokenizer` emits punctuation and symbols as tokens. Both counts are retained as an audit of
tokenisation, while the manuscript's word and lexical statistics exclude punctuation-only and
symbol-only tokens.

In [ ]:
from shekar import WordTokenizer, Normalizer

tokenizer = WordTokenizer()


def is_nonword_token(token: str) -> bool:
    "True for tokens made only of Unicode punctuation or symbols."
    chars = [ch for ch in token if not ch.isspace()]
    return bool(chars) and all(unicodedata.category(ch)[0] in {"P", "S"} for ch in chars)


df["tokens"] = df["text"].map(lambda t: list(tokenizer.tokenize(t)))
df["words"] = df["tokens"].map(lambda ts: [t for t in ts if not is_nonword_token(t)])
df["n_tokens"] = df["tokens"].str.len()
df["n_words"] = df["words"].str.len()
df["words_per_sec"] = df["n_words"] / df["duration"]

token_freq = Counter(t for ts in df["tokens"] for t in ts)
word_freq = Counter(w for ws in df["words"] for w in ws)
char_freq = Counter("".join(df["text"]))
hapax = sum(1 for n in word_freq.values() if n == 1)

print("Tokens and words")
print(f"  tokens (incl. punctuation) {df.n_tokens.sum():,}   types {len(token_freq):,}")
print(f"  words  (excl. punctuation/symbols) {df.n_words.sum():,}   types {len(word_freq):,}")
print(f"  mean tokens per clip       {df.n_tokens.mean():.2f}")
print(
    f"  mean words per clip        {df.n_words.mean():.2f}  "
    f"(median {df.n_words.median():.0f}, range {df.n_words.min()}-{df.n_words.max()})"
)
print(f"  mean speaking rate         {df.words_per_sec.mean():.2f} words/s")

print("\nLexical diversity")
print(f"  type-token ratio (words)   {len(word_freq) / df.n_words.sum():.4f}")
print(f"  hapax legomena             {hapax:,} ({100 * hapax / len(word_freq):.1f}% of word types)")
print(
    f"  top 100 types cover        {100 * sum(n for _, n in word_freq.most_common(100)) / df.n_words.sum():.1f}% of tokens"
)

print("\nCharacters")
print(f"  characters (with spaces)   {df.n_chars.sum():,}")
print(f"  mean characters per clip   {df.n_chars.mean():.1f}")
print(f"  character vocabulary       {len(char_freq)} distinct")

# Punctuation policy: transcripts keep punctuation and standalone symbols, so they are reported
# rather than stripped. ZWNJ is orthographic in Persian, not punctuation.
ZWNJ = "‌"
punct_tokens = Counter(t for ts in df["tokens"] for t in ts if is_nonword_token(t))
print("\nPunctuation and orthography")
print(
    f"  punctuation/symbol tokens  {sum(punct_tokens.values()):,} "
    f"({100 * sum(punct_tokens.values()) / df.n_tokens.sum():.1f}% of tokens)"
)
print(
    f"  clips ending in a mark     {100 * df.text.str.rstrip().str[-1].isin(list('.!?؟…')).mean():.1f}%"
)
print(f"  clips containing ZWNJ      {100 * df.text.str.contains(ZWNJ).mean():.1f}%")
print(f"  ZWNJ occurrences           {int(df.text.str.count(ZWNJ).sum()):,}")
print(
    "  most frequent marks:      ",
    ", ".join(f"{p!r} {n:,}" for p, n in punct_tokens.most_common(6)),
)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.histplot(
    df, x="n_words", bins=range(0, int(df.n_words.quantile(0.999)) + 2), ax=ax, edgecolor="none"
)
ax.axvline(
    df.n_words.mean(), color="crimson", ls="--", lw=1.4, label=f"mean {df.n_words.mean():.1f}"
)
ax.legend(frameon=False)
ax.set(xlabel="words per clip", ylabel="clips")
fig.tight_layout()
save(fig, "04a_words_per_clip")
plt.show()
plt.close(fig)

sample = df.sample(min(8000, len(df)), random_state=0)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.scatterplot(
    sample,
    x="duration",
    y="n_words",
    hue="register",
    palette=REGISTER_COLORS,
    s=10,
    alpha=0.35,
    edgecolor="none",
    ax=ax,
)
sns.regplot(
    data=df,
    x="duration",
    y="n_words",
    scatter=False,
    color="black",
    line_kws={"lw": 1.3, "ls": "--"},
    ax=ax,
)
ax.set(xlabel="duration (s)", ylabel="words per clip")
ax.legend(title="", frameon=False)
fig.tight_layout()
save(fig, "04b_words_duration", "words_duration")
plt.show()
plt.close(fig)

freqs = np.array(sorted(word_freq.values(), reverse=True))
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.loglog(np.arange(1, len(freqs) + 1), freqs, lw=1.6)
ax.set(xlabel="word-type rank", ylabel="frequency")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
save(fig, "04c_rank_frequency", "rank_frequency")
plt.show()
plt.close(fig)

print(
    f"Pearson r between duration and word count: {df[['duration', 'n_words']].corr().iloc[0, 1]:.3f}"
)

## 6. Prompt reuse and overlap between splits

Splits are speaker-disjoint but not text-disjoint. Where a test prompt also occurs in training, a
fine-tuned model has already seen the reference transcription. Quoted in the manuscript, so
computed here.

In [ ]:
prompts = {s: set(df.loc[df.split == s, "text"]) for s in SPLIT_ORDER}

print(
    f"{len(df):,} clips from {df.text.nunique():,} distinct prompts "
    f"(mean {len(df) / df.text.nunique():.2f} readings per prompt, "
    f"max {df.text.value_counts().max()})"
)
print()
print("distinct prompts per split:", {s: f"{len(p):,}" for s, p in prompts.items()})
print()

overlap = []
for split in ("validation", "test"):
    shared = prompts[split] & prompts["train"]
    clips = df[(df.split == split) & (df.text.isin(prompts["train"]))]
    n_split = int((df.split == split).sum())
    overlap.append(
        {
            "split": split,
            "prompts shared with train": len(shared),
            "% of split prompts": 100 * len(shared) / len(prompts[split]),
            "clips affected": len(clips),
            "% of split clips": 100 * len(clips) / n_split,
        }
    )

overlap = pd.DataFrame(overlap)
print(overlap.round(1).to_string(index=False))
print(
    "\nEvaluation on these partitions is not text-disjoint; re-partition on unique"
    "\ntranscripts if a leakage-free measurement is required."
)

In [ ]:
# Vocabulary coverage: how much of an evaluation split's running text is made of
# word types the training split never shows.
train_vocab = Counter(w for ws in df.loc[df.split == "train", "words"] for w in ws)

coverage = []
for split in ("validation", "test"):
    words = [w for ws in df.loc[df.split == split, "words"] for w in ws]
    types = set(words)
    oov_tokens = sum(1 for w in words if w not in train_vocab)
    coverage.append(
        {
            "split": split,
            "word tokens": len(words),
            "word types": len(types),
            "OOV types": len(types - set(train_vocab)),
            "OOV type rate (%)": 100 * len(types - set(train_vocab)) / len(types),
            "OOV token rate (%)": 100 * oov_tokens / len(words),
        }
    )

print(f"training vocabulary: {len(train_vocab):,} word types")
print()
print(pd.DataFrame(coverage).round(2).to_string(index=False))

## 7. Named-entity annotation

The released `shekar` interface processes one document at a time. The notebook retains that path
for POS tagging, whose exported graph has a fixed batch axis, and uses a small compatibility wrapper
to run NER in dynamically padded batches. NER is forced to the CPU execution provider because it was
faster for these short texts on the machine used here. Both passes are checkpointed and resumable.

In [ ]:
from shekar import NER
import onnxruntime as ort


def annotate(texts, cache: Path = ANNOT_CACHE, checkpoint_every: int = 2500) -> pd.DataFrame:
    """Tag every transcript with shekar's NER model.

    Calls the taggers one document at a time, matching shekar's own call path so
    the results are reproducible from the released library. An interrupted run
    costs at most `checkpoint_every` documents rather than the whole pass.
    """
    from neyshekar_experiments.protocol import digest
    import importlib.metadata

    cache.parent.mkdir(parents=True, exist_ok=True)
    signature = {
        "texts_sha256": digest(list(texts)),
        "shekar_version": importlib.metadata.version("shekar"),
    }
    signature_path = cache.with_suffix(".meta.json")
    partial_path = cache.with_name(cache.stem + ".partial.parquet")
    if cache.exists() or partial_path.exists():
        if not signature_path.exists() or json.loads(signature_path.read_text()) != signature:
            raise ValueError("Stale annotation cache; remove it and rerun annotation")
    signature_path.write_text(json.dumps(signature, sort_keys=True))
    if cache.exists():
        print(f"loading cached annotations from {cache.relative_to(ROOT)}")
        out = pd.read_parquet(cache)
        if len(out) != len(texts):
            raise ValueError(f"annotation cache has {len(out):,} rows for {len(texts):,} texts")
        return out

    columns = ["ner_json"]
    partial = cache.with_name(cache.stem + ".partial.parquet")
    rows = []
    if partial.exists():
        rows = pd.read_parquet(partial)[columns].to_records(index=False).tolist()
        print(f"resuming from {len(rows):,} rows in {partial.name}", flush=True)

    if len(rows) >= len(texts):
        out = pd.DataFrame(rows[: len(texts)], columns=columns)
        out.to_parquet(cache, index=False)
        partial.unlink(missing_ok=True)
        return out

    print("onnxruntime providers:", ort.get_available_providers(), flush=True)
    ner = NER()
    ner(texts[0])  # warm up before timing the loop

    t0, done0 = time.time(), len(rows)
    for i, text in enumerate(texts[len(rows) :], len(rows) + 1):
        rows.append((json.dumps(ner(text), ensure_ascii=False),))
        if i % checkpoint_every == 0:
            pd.DataFrame(rows, columns=columns).to_parquet(partial, index=False)
            rate = (i - done0) / (time.time() - t0)
            # stderr so progress is visible while the cell is still running
            print(
                f"  {i:,}/{len(texts):,}  ({rate:.0f} docs/s, "
                f"{(len(texts) - i) / rate / 60:.1f} min remaining)",
                file=sys.stderr,
                flush=True,
            )

    out = pd.DataFrame(rows, columns=columns)
    out.to_parquet(cache, index=False)
    partial.unlink(missing_ok=True)
    print(f"cached -> {cache.relative_to(ROOT)} in {(time.time() - t0) / 60:.1f} min")
    return out


annot = annotate(df["text"].tolist())
df["entities"] = annot["ner_json"].map(json.loads).values
df["n_entities"] = df["entities"].str.len()

print(f"\n{df.n_entities.sum():,} entity mentions across the corpus")

## 8. Named entities

The second design axis: mention total, distribution by type, coverage, and density by register.

In [ ]:
ENTITY_NAMES = {
    "LOC": "location",
    "PER": "person",
    "ORG": "organisation",
    "DAT": "date",
    "EVE": "event",
}

ent_rows = pd.DataFrame(
    [(i, span, label) for i, ents in zip(df.index, df["entities"]) for span, label in ents],
    columns=["row", "span", "label"],
)

by_type = ent_rows["label"].value_counts().rename_axis("label").reset_index(name="mentions")
by_type["type"] = by_type["label"].map(lambda label: ENTITY_NAMES.get(label, label.lower()))
by_type["share (%)"] = 100 * by_type["mentions"] / len(ent_rows)

print(f"entity mentions        {len(ent_rows):,}")
print(f"distinct surface forms {ent_rows.span.nunique():,}")
print(f"mentions per clip      {df.n_entities.mean():.2f}")
print(f"clips with >=1 entity  {100 * (df.n_entities > 0).mean():.1f}%")
print(f"mentions per 100 words {100 * len(ent_rows) / df.n_words.sum():.2f}")
print()
print(by_type[["type", "label", "mentions", "share (%)"]].round(2).to_string(index=False))
print()
print("Twenty most frequent surface forms")
print(ent_rows["span"].value_counts().head(20).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.barplot(by_type, y="type", x="mentions", color=CATEGORICAL[0], ax=ax)
for c in ax.containers:
    ax.bar_label(c, fmt="{:,.0f}", padding=3, fontsize=9)
ax.set(xlabel="mentions", ylabel="")
ax.set_xlim(0, by_type["mentions"].max() * 1.18)
fig.tight_layout()
save(fig, "05a_entity_types", "entity_types")
plt.show()
plt.close(fig)

dist = df["n_entities"].clip(upper=6).value_counts().sort_index()
dist.index = [str(i) if i < 6 else "6+" for i in dist.index]
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.barplot(x=dist.index, y=dist.values, color=CATEGORICAL[0], ax=ax)
ax.set(xlabel="entity mentions per clip", ylabel="clips")
fig.tight_layout()
save(fig, "05b_entities_per_clip")
plt.show()
plt.close(fig)

density = (
    df.groupby("register", observed=True)
    .apply(lambda g: 100 * g.n_entities.sum() / g.n_words.sum(), include_groups=False)
    .rename("per 100 words")
    .reset_index()
)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.barplot(
    density,
    x="register",
    y="per 100 words",
    hue="register",
    palette=REGISTER_COLORS,
    legend=False,
    ax=ax,
)
for c in ax.containers:
    ax.bar_label(c, fmt="{:.2f}", padding=2, fontsize=9)
ax.set(xlabel="", ylabel="mentions per 100 words")
fig.tight_layout()
save(fig, "05c_entity_density_register", "entity_density_register")
plt.show()
plt.close(fig)

## 9. Collection evidence

Speaker/demographic counts below are historical platform reports, not independently verified identity records. Counts summing to 198 do not prove speaker disjointness. The verification module requires actual clip-to-speaker assignments. Agreement is recomputed from the exported pseudonymous item-level labels.

### Prompt sources

Three broad source families with different jobs: a homograph corpus, human-written phrases, and
language-model output. The chart separates the language-model share by model. Proportions are
approximate and come from the collection platform.

In [ ]:
# Approximate composition of the prompt pool, recorded by the collection platform.
PROMPT_SOURCES = {
    "Claude Sonnet 4.5": 36,
    "GPT-4.5 generated": 27,
    "GPT-5.1 generated": 19,
    "Human-written": 13,
    "HomoRich": 5,
}
PROMPT_COLORS = ["#0173b2", "#de8f05", "#029e73", "#d55e00", "#cc78bc"]

assert sum(PROMPT_SOURCES.values()) == 100, "prompt source shares must total 100"

fig, ax = plt.subplots(figsize=(8.2, 5.2))
wedges, _, autotexts = ax.pie(
    PROMPT_SOURCES.values(),
    colors=PROMPT_COLORS,
    startangle=90,
    counterclock=False,
    autopct=lambda v: f"{v:.0f}%",
    pctdistance=0.72,
    # a 2px surface gap between segments keeps the boundaries readable
    wedgeprops={"edgecolor": SURFACE, "linewidth": 2},
    textprops={"fontsize": 11},
)
for t in autotexts:
    t.set_color(SURFACE)
    t.set_fontweight("bold")

# Direct labels rather than a legend: identity is never carried by colour alone,
# which is also the relief the contrast check requires for the lighter hues.
ax.legend(
    wedges,
    [k.replace("\n", " ") for k in PROMPT_SOURCES],
    loc="center left",
    bbox_to_anchor=(0.98, 0.5),
    frameon=False,
    fontsize=11,
)
ax.set(aspect="equal")

save(fig, "06_prompt_sources", "prompt_sources")
plt.show()
plt.close(fig)

print("\n".join(f"{k.replace(chr(10), ' '):38s} {v:3d}%" for k, v in PROMPT_SOURCES.items()))

In [ ]:
# Read from the collection platform's admin dashboard on 2026-09-01. The released
# corpus carries no speaker or demographic fields, so these cannot be recomputed
# from the data and are recorded here as the single point of provenance.
PLATFORM = {
    "prompts_in_pool": 36_554,
    "recordings_submitted": 69_744,
    "recordings_accepted": 62_279,
    "recordings_rejected": 7_465,
    "contributors": 198,
    "accepted_male": 28_076,
    "accepted_female": 33_735,
}

# speakers per split: (total, male, female)
SPEAKERS = {"train": (142, 53, 67), "validation": (26, 12, 13), "test": (30, 16, 13)}

print("Collection totals (platform)")
print(f"  prompts in pool          {PLATFORM['prompts_in_pool']:,}")
print(f"  recordings submitted     {PLATFORM['recordings_submitted']:,}")
print(
    f"  accepted                 {PLATFORM['recordings_accepted']:,} "
    f"({100 * PLATFORM['recordings_accepted'] / PLATFORM['recordings_submitted']:.1f}%)"
)
print(
    f"  rejected                 {PLATFORM['recordings_rejected']:,} "
    f"({100 * PLATFORM['recordings_rejected'] / PLATFORM['recordings_submitted']:.1f}%)"
)

assert (
    PLATFORM["recordings_accepted"] + PLATFORM["recordings_rejected"]
    == PLATFORM["recordings_submitted"]
), "accepted + rejected != submitted"
assert PLATFORM["recordings_accepted"] == len(df), "platform accepted count != released clips"
print(f"\n  accepted count matches the {len(df):,} clips in the release")

speakers = pd.DataFrame(
    [(s, t, m, f, t - m - f) for s, (t, m, f) in SPEAKERS.items()],
    columns=["split", "speakers", "male", "female", "unspecified"],
).set_index("split")
speakers.loc["full"] = speakers.sum()

print("\nSpeakers per split")
print(speakers.to_string())

from neyshekar_experiments.validation import verify_all

verification = verify_all()
print("\nSpeaker verification:", verification["speakers"])
print("Historical dashboard counts are descriptive; they do not establish disjointness.")

gendered = PLATFORM["accepted_male"] + PLATFORM["accepted_female"]
print(
    f"\nAccepted clips by speaker gender: {PLATFORM['accepted_male']:,} male, "
    f"{PLATFORM['accepted_female']:,} female, "
    f"{len(df) - gendered:,} unspecified ({100 * (len(df) - gendered) / len(df):.1f}%)"
)

In [ ]:
# Recompute agreement from item-level decisions, preserving all votes per item.
from neyshekar_experiments.validation import PUBLIC, reliability

labels = pd.read_csv(PUBLIC / "rater_labels.csv")
shared = labels[labels.cohort.eq("shared")]
agreement = reliability(shared, replicates=20_000, seed=42)
print(f"{agreement['items']} items, {agreement['raters']} raters, {agreement['labels']} votes")
print("95% CIs: percentile bootstrap resampling complete items")
for metric in ("raw_agreement", "accept_share", "fleiss_kappa", "gwet_ac1"):
    print(metric, agreement[metric])
print("Accept-specific agreement:", agreement["accept_agreement"])
print("Reject-specific agreement:", agreement["reject_agreement"])
print(
    "Snapshot items:",
    labels.item_id.nunique(),
    "| multiply reviewed:",
    int(labels.groupby("item_id").size().ge(2).sum()),
)
print("The export snapshot has one additional reviewed item relative to the old dashboard totals.")

## 10. Main dataset table

Scale, speakers and composition, overall and per partition. Speaker rows come from the constants
above; everything else is derived.

In [ ]:
def column(frame: pd.DataFrame, split: str | None) -> dict:
    "One column of the main table: derived rows from `frame`, speaker rows from SPEAKERS."
    words = [w for ws in frame["words"] for w in ws]
    if split is None:
        total, male, female = (sum(v[i] for v in SPEAKERS.values()) for i in range(3))
    else:
        total, male, female = SPEAKERS[split]
    return {
        "Utterances": f"{len(frame):,}",
        "Duration (h)": f"{frame.duration.sum() / 3600:.2f}",
        "Speakers": f"{total:,}",
        "Female speakers": f"{female:,}",
        "Male speakers": f"{male:,}",
        "Unspecified speakers": f"{total - male - female:,}",
        "Mean duration (s)": f"{frame.duration.mean():.2f}",
        "Median duration (s)": f"{frame.duration.median():.2f}",
        "Words": f"{len(words):,}",
        "Unique words": f"{len(set(words)):,}",
        "Distinct prompts": f"{frame.text.nunique():,}",
        "Informal speech (%)": f"{100 * (frame.register == 'informal').mean():.2f}",
    }


main_table = pd.DataFrame(
    {
        "Full": column(df, None),
        "Train": column(df[df.split == "train"], "train"),
        "Validation": column(df[df.split == "validation"], "validation"),
        "Test": column(df[df.split == "test"], "test"),
    }
)
main_table.index.name = "Statistic"
main_table

## 11. Comparison with Persian Common Voice

The template this corpus departs from, so the reference for all three axes. Uses the validated
portion of release 26.0, read from its own TSVs. Entity density is scored on a size-matched sample
and reported per 100 words, since Common Voice clips are shorter and a per-clip rate would
confound richness with length.

In [ ]:
CV_DIR = DATA / "cv26" / "cv-corpus-26.0-2026-06-12" / "fa"
CV_NER_CACHE = ROOT / "results/annotations/cv26_ner_sample.parquet"


def load_common_voice(cv_dir: Path = CV_DIR) -> pd.DataFrame:
    "Validated Common Voice clips with per-clip duration attached."
    if not cv_dir.exists():
        raise FileNotFoundError(
            f"{cv_dir} not found. Extract the TSVs from the Common Voice archive with:\n"
            f"  tar -xzf <cv-corpus-*.tar.gz> -C {DATA} --wildcards '*.tsv'"
        )
    cv = pd.read_csv(cv_dir / "validated.tsv", sep="\t", quoting=3, low_memory=False)
    dur = pd.read_csv(cv_dir / "clip_durations.tsv", sep="\t", quoting=3)
    dur.columns = ["path", "ms"]
    cv = cv.merge(dur, on="path", how="left", validate="one_to_one")
    cv["duration"] = cv["ms"] / 1000.0
    cv = cv[cv.sentence.notna() & cv.duration.notna()].copy()
    cv["sentence"] = cv["sentence"].astype(str)
    return cv


cv = load_common_voice()
normalizer = Normalizer()
cv["normalised_sentence"] = cv["sentence"].map(normalizer)
cv["words"] = [
    [w for w in tokenizer.tokenize(s) if not is_nonword_token(w)] for s in cv.normalised_sentence
]
cv["n_words"] = cv["words"].str.len()
cv["informal"] = [informal_clf(s)[1] == 1 for s in cv.normalised_sentence]

cv_word_freq = Counter(w for ws in cv["words"] for w in ws)


# Vocabulary size is sensitive both to corpus size and to orthographic
# normalisation, so a raw type count or a type-token ratio would compare two
# different things: type-token ratio falls mechanically as a corpus grows, and
# Neyshekar transcripts are normalised while Common Voice sentences are not.
# The comparable quantity puts both sides through the same normaliser and draws
# the same number of tokens from each.
def vocabulary_at(tokens, n_tokens, draws=5, seed=0):
    "Mean distinct types in `n_tokens` sampled without replacement."
    rng = np.random.default_rng(seed)
    pool = np.array(tokens, dtype=object)
    if len(pool) < n_tokens:
        return float(len(set(pool)))
    return float(
        np.mean([len(set(rng.choice(pool, n_tokens, replace=False))) for _ in range(draws)])
    )


cv_norm_words = [w for ws in cv["words"] for w in ws]
ney_all_words = [w for ws in df["words"] for w in ws]
MATCHED_TOKENS = len(ney_all_words)
ney_vocab_matched = len(set(ney_all_words))
cv_vocab_matched = vocabulary_at(cv_norm_words, MATCHED_TOKENS)
print(
    f"vocabulary at {MATCHED_TOKENS:,} tokens (both normalised): "
    f"Neyshekar {ney_vocab_matched:,}, Common Voice {cv_vocab_matched:,.0f}"
)
print(f"Common Voice 26.0 (fa, validated): {len(cv):,} clips, {cv.duration.sum() / 3600:.2f} h")

In [ ]:
def infer_entity_batch(ner, texts: list[str]) -> list[list]:
    "Run one dynamically padded ONNX batch while preserving shekar's aggregation."
    model = ner.model
    model.tokenizer.enable_padding = False
    encoded = [model.tokenizer(text) for text in texts]
    # Sentence prompts should occupy one model window. Fall back to shekar's
    # overflow-aware path if that invariant ever changes.
    if any(enc["input_ids"].shape[0] != 1 for enc in encoded):
        return [ner(text) for text in texts]

    max_len = max(enc["input_ids"].shape[1] for enc in encoded)
    input_ids = np.full((len(texts), max_len), model.tokenizer.pad_token_id, dtype=np.int64)
    attention_mask = np.zeros((len(texts), max_len), dtype=np.int64)
    for row, enc in enumerate(encoded):
        length = enc["input_ids"].shape[1]
        input_ids[row, :length] = enc["input_ids"][0]
        attention_mask[row, :length] = enc["attention_mask"][0]

    logits = model.session.run(None, {"input_ids": input_ids, "attention_mask": attention_mask})[0]
    pred_ids = np.argmax(logits, axis=-1)
    special_ids = {
        model.tokenizer.pad_token_id,
        model.tokenizer.cls_token_id,
        model.tokenizer.sep_token_id,
    }
    results = []
    for ids, mask, tags in zip(input_ids, attention_mask, pred_ids):
        valid = [i for i in range(max_len) if mask[i] and int(ids[i]) not in special_ids]
        tokens = [model.tokenizer.id_to_token(int(ids[i])) for i in valid]
        results.append(model._aggregate_entities(tokens, tags[valid]))
    return results


def score_entities(texts, cache: Path, checkpoint_every: int = 2500, batch_size: int = 32) -> list:
    "Run shekar NER over `texts`, batching and caching the expensive pass."
    from neyshekar_experiments.protocol import digest
    import importlib.metadata

    cache.parent.mkdir(parents=True, exist_ok=True)
    signature = {
        "texts_sha256": digest(list(texts)),
        "shekar_version": importlib.metadata.version("shekar"),
    }
    signature_path = cache.with_suffix(".meta.json")
    partial_path = cache.with_name(cache.stem + ".partial.parquet")
    if cache.exists() or partial_path.exists():
        if not signature_path.exists() or json.loads(signature_path.read_text()) != signature:
            raise ValueError("Stale annotation cache; remove it and rerun annotation")
    signature_path.write_text(json.dumps(signature, sort_keys=True))
    if cache.exists():
        print(f"loading cached entity scores from {cache.relative_to(ROOT)}")
        cached_rows = pd.read_parquet(cache)["ner_json"]
        if len(cached_rows) != len(texts):
            raise ValueError(f"entity cache has {len(cached_rows):,} rows for {len(texts):,} texts")
        return [json.loads(x) for x in cached_rows]

    partial = cache.with_name(cache.stem + ".partial.parquet")
    rows = []
    if partial.exists():
        rows = pd.read_parquet(partial)["ner_json"].tolist()
        print(f"resuming from {len(rows):,} rows", flush=True)

    ner = NER()
    ner.model.session.set_providers(["CPUExecutionProvider"])
    infer_entity_batch(ner, texts[:1])
    t0, done0 = time.time(), len(rows)
    next_checkpoint = ((len(rows) // checkpoint_every) + 1) * checkpoint_every
    for start in range(len(rows), len(texts), batch_size):
        batch = texts[start : start + batch_size]
        rows.extend(json.dumps(x, ensure_ascii=False) for x in infer_entity_batch(ner, batch))
        i = len(rows)
        if i >= next_checkpoint:
            pd.DataFrame({"ner_json": rows}).to_parquet(partial, index=False)
            rate = (i - done0) / (time.time() - t0)
            print(
                f"  {i:,}/{len(texts):,} ({rate:.0f} docs/s, "
                f"{(len(texts) - i) / rate / 60:.1f} min remaining)",
                file=sys.stderr,
                flush=True,
            )
            next_checkpoint += checkpoint_every

    pd.DataFrame({"ner_json": rows}).to_parquet(cache, index=False)
    partial.unlink(missing_ok=True)
    print(f"cached -> {cache.relative_to(ROOT)} in {(time.time() - t0) / 60:.1f} min")
    return [json.loads(x) for x in rows]


# A sample matched to Neyshekar's clip count keeps the two density figures
# comparable without scoring all 341k Common Voice sentences.
cv_sample = cv.sample(min(len(df), len(cv)), random_state=0).reset_index(drop=True)
cv_sample["entities"] = score_entities(cv_sample.normalised_sentence.tolist(), CV_NER_CACHE)
cv_sample["n_entities"] = cv_sample["entities"].str.len()

cv_density = 100 * cv_sample.n_entities.sum() / cv_sample.n_words.sum()
ney_density = 100 * len(ent_rows) / df.n_words.sum()
print(
    f"\nentity density per 100 words — Neyshekar {ney_density:.2f}, Common Voice {cv_density:.2f}"
)

In [ ]:
def gender_counts(series) -> tuple[int, int]:
    "Common Voice records gender as male_masculine / female_feminine."
    g = series.fillna("unspecified").astype(str)
    return int(g.str.startswith("male").sum()), int(g.str.startswith("female").sum())


cv_male, cv_female = gender_counts(cv.gender)
ney_female, ney_male = PLATFORM["accepted_female"], PLATFORM["accepted_male"]

comparison = pd.DataFrame(
    {
        "Neyshekar": {
            "Clips": f"{len(df):,}",
            "Duration (h)": f"{df.duration.sum() / 3600:.2f}",
            "Mean clip duration (s)": f"{df.duration.mean():.2f}",
            "Median clip duration (s)": f"{df.duration.median():.2f}",
            "Clips over 10 s (%)": f"{100 * (df.duration > 10).mean():.1f}",
            "Clips under 4 s (%)": f"{100 * (df.duration < 4).mean():.1f}",
            "Mean words per clip": f"{df.n_words.mean():.2f}",
            "Word tokens": f"{df.n_words.sum():,}",
            "Word types": f"{len(word_freq):,}",
            "Word types at matched token count": f"{ney_vocab_matched:,}",
            "Informal clips (%)": f"{100 * (df.register == 'informal').mean():.2f}",
            "Entity mentions per 100 words": f"{ney_density:.2f}",
            "Speakers": f"{PLATFORM['contributors']:,}",
            "Clips by female speakers (%)": f"{100 * ney_female / len(df):.1f}",
            "Clips by male speakers (%)": f"{100 * ney_male / len(df):.1f}",
            "Distinct prompts": f"{df.text.nunique():,}",
            "Readings per prompt": f"{len(df) / df.text.nunique():.2f}",
        },
        "Common Voice 26.0": {
            "Clips": f"{len(cv):,}",
            "Duration (h)": f"{cv.duration.sum() / 3600:.2f}",
            "Mean clip duration (s)": f"{cv.duration.mean():.2f}",
            "Median clip duration (s)": f"{cv.duration.median():.2f}",
            "Clips over 10 s (%)": f"{100 * (cv.duration > 10).mean():.1f}",
            "Clips under 4 s (%)": f"{100 * (cv.duration < 4).mean():.1f}",
            "Mean words per clip": f"{cv.n_words.mean():.2f}",
            "Word tokens": f"{cv.n_words.sum():,}",
            "Word types": f"{len(cv_word_freq):,}",
            "Word types at matched token count": f"{cv_vocab_matched:,.0f}",
            "Informal clips (%)": f"{100 * cv.informal.mean():.2f}",
            "Entity mentions per 100 words": f"{cv_density:.2f}",
            "Speakers": f"{cv.client_id.nunique():,}",
            "Clips by female speakers (%)": f"{100 * cv_female / len(cv):.1f}",
            "Clips by male speakers (%)": f"{100 * cv_male / len(cv):.1f}",
            "Distinct prompts": f"{cv.sentence.nunique():,}",
            "Readings per prompt": f"{len(cv) / cv.sentence.nunique():.2f}",
        },
    }
)
comparison.index.name = "Statistic"
comparison

In [ ]:
NEY, CVC = CATEGORICAL[0], CATEGORICAL[1]

# Values outside the displayed ranges are omitted rather than clipped into the
# last bin, which would create an artificial spike at the right boundary.
bins = np.arange(0, 20.5, 0.5)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for frame, col, label in [(cv, CVC, "Common Voice 26.0"), (df, NEY, "Neyshekar v6")]:
    shown = frame.loc[frame.duration <= 20, "duration"]
    ax.hist(
        shown,
        bins=bins,
        weights=np.full(len(shown), 1 / len(frame)),
        color=col,
        alpha=0.66,
        label=label,
    )
ax.set(xlabel="duration (s)", ylabel="proportion of all clips")
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "07a_cv_duration_histogram", "cv_duration_distribution")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
sns.ecdfplot(x=cv.duration, ax=ax, color=CVC, lw=2, label="Common Voice 26.0")
sns.ecdfplot(x=df.duration, ax=ax, color=NEY, lw=2, label="Neyshekar v6")
ax.axvline(10, color="#52514e", ls=":", lw=1.2)
ax.set(xlabel="duration (s)", ylabel="cumulative proportion", xlim=(0, 20))
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "07b_cv_duration_ecdf")
plt.show()
plt.close(fig)

wb = np.arange(0, 31)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for frame, col, label in [(cv, CVC, "Common Voice 26.0"), (df, NEY, "Neyshekar v6")]:
    shown = frame.loc[frame.n_words <= 30, "n_words"]
    ax.hist(
        shown,
        bins=wb,
        weights=np.full(len(shown), 1 / len(frame)),
        color=col,
        alpha=0.66,
        label=label,
    )
ax.set(xlabel="words per clip", ylabel="proportion of all clips")
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "07c_cv_words_histogram", "cv_words_distribution")
plt.show()
plt.close(fig)

## 12. Summary tables

Figures quoted in the descriptor. LaTeX is written to `paper/` so the manuscript tables are
regenerated rather than transcribed.

In [ ]:
def pct(mask):
    return f"{mask.sum():,} ({100 * mask.mean():.2f}%)"


summary = pd.DataFrame(
    [
        ("Validated clips", f"{len(df):,}"),
        ("Total duration (hours)", f"{df.duration.sum() / 3600:.2f}"),
        ("Mean clip duration (seconds)", f"{df.duration.mean():.2f}"),
        ("Median clip duration (seconds)", f"{df.duration.median():.2f}"),
        ("Tokens (including punctuation)", f"{df.n_tokens.sum():,}"),
        ("Token types (including punctuation)", f"{len(token_freq):,}"),
        ("Word tokens (excluding punctuation/symbols)", f"{df.n_words.sum():,}"),
        ("Word types (excluding punctuation/symbols)", f"{len(word_freq):,}"),
        ("Distinct prompts", f"{df.text.nunique():,}"),
        ("Formal clips", pct(df.register == "formal")),
        ("Informal clips", pct(df.register == "informal")),
        ("Named-entity mentions", f"{len(ent_rows):,}"),
        ("Distinct named entities", f"{ent_rows.span.nunique():,}"),
        ("Clips with at least one entity", f"{100 * (df.n_entities > 0).mean():.1f}%"),
        (
            "Training split",
            f"{(df.split == 'train').sum():,} clips ({df.loc[df.split == 'train', 'duration'].sum() / 3600:.2f} h)",
        ),
        (
            "Validation split",
            f"{(df.split == 'validation').sum():,} clips ({df.loc[df.split == 'validation', 'duration'].sum() / 3600:.2f} h)",
        ),
        (
            "Test split",
            f"{(df.split == 'test').sum():,} clips ({df.loc[df.split == 'test', 'duration'].sum() / 3600:.2f} h)",
        ),
    ],
    columns=["Quantity", "Value"],
)

summary

In [ ]:
def to_latex(body_rows, caption, colspec, header) -> str:
    "Render a table as standalone LaTeX, with no dependency on a .bib or style file."
    return "\n".join(
        [
            r"\begin{table}[t]",
            r"\centering",
            rf"\caption{{{caption}}}",
            rf"\begin{{tabular}}{{{colspec}}}",
            r"\hline",
            header,
            r"\hline",
            *body_rows,
            r"\hline",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )


def esc(v):
    return str(v).replace("%", r"\%")


main_rows = [
    " & ".join([esc(i)] + [esc(v) for v in r]) + r" \\"
    for i, r in zip(main_table.index, main_table.to_numpy())
]
table1 = to_latex(
    main_rows,
    "Composition of the Neyshekar corpus overall and by predefined partition. "
    "Speaker counts are recorded by the collection platform; the release itself "
    "carries no speaker identifiers.",
    "lrrrr",
    "Statistic & Full & Train & Validation & Test " + r"\\",
)

detail_rows = [rf"{q} & {esc(v)} \\" for q, v in summary.itertuples(index=False)]
table2 = to_latex(
    detail_rows,
    "Transcript, register and named-entity statistics of the Neyshekar corpus.",
    "lr",
    r"Quantity & Value \\",
)

comp_rows = [
    " & ".join([esc(i)] + [esc(v) for v in r]) + r" \\"
    for i, r in zip(comparison.index, comparison.to_numpy())
]
table3 = to_latex(
    comp_rows,
    "Neyshekar v6 compared with the validated portion of the Persian subset of "
    "Mozilla Common Voice 26.0. Entity density is scored on a random sample of "
    "Common Voice sentences matched in size to Neyshekar v6.",
    "lrr",
    r"Statistic & Neyshekar v6 & Common Voice 26.0 \\",
)

paper = ROOT / "acl"
paper.mkdir(exist_ok=True)
(paper / "table1_generated.tex").write_text(table1, encoding="utf-8")
(paper / "table2_generated.tex").write_text(table2, encoding="utf-8")
(paper / "table3_generated.tex").write_text(table3, encoding="utf-8")
print("wrote acl/table1_generated.tex, table2_generated.tex and table3_generated.tex\n")
print(table1)